In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Mon Aug 31 14:04:17 PDT 2026


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, gbd_data

In [3]:
from lsff_utils import gbd_data

In [4]:
location = "india"
vehicle = "rice"

In [5]:
intervention_scenarios = config_utils.get_config()["custom_intervention_scenarios"].get(
    location, ["intervention"]
)
intervention_scenarios

['intervention']

In [6]:
# Get most recent GBD year, used by calls in gbd_data
YEAR = gbd_data.most_recent_year()
YEAR

2023

## Forecasted births and stillbirths

In [7]:
# NOTE: Year must match base year used in TFR rescaling below, so we
# keep it at 2022 for now
# TODO: Update year from 2022 (to 2023 to match GBD?) if we can replace
# TFR data 
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate,
        "estimate",
        location.title(),
        years=2022,
    ).value

In [8]:
# Filter out lower and upper values, keep mean only
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr.loc[asfr>0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2022        2023        0.000356
                  15.0       20.0     2022        2023        0.009175
                  20.0       25.0     2022        2023        0.099802
                  25.0       30.0     2022        2023        0.126981
                                                                ...   
                  35.0       40.0     2022        2023        0.033608
                  40.0       45.0     2022        2023        0.011475
                  45.0       50.0     2022        2023        0.003128
                  50.0       55.0     2022        2023        0.000284
Name: value, Length: 9, dtype: float64

In [9]:
# TODO: Update this with more recent GBD data. Also, does this belong in
# 0100_data_prep? Maybe not, if it's only used in this notebook.
# NOTE: The year in the denominator here must match the year of the ASFR
# call above. The year in the numerator is the target year we want data
# for.
# Scale ASFR in each category down proportionally to the scale-down in
# total fertility rate (TFR) forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = 1.61 / 1.91  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2030_to_2022_ratio = 4.43 / 4.96  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = 3.27 / 4.10  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2022        2023        0.000300
                  15.0       20.0     2022        2023        0.007734
                  20.0       25.0     2022        2023        0.084126
                  25.0       30.0     2022        2023        0.107037
                                                                ...   
                  35.0       40.0     2022        2023        0.028329
                  40.0       45.0     2022        2023        0.009673
                  45.0       50.0     2022        2023        0.002636
                  50.0       55.0     2022        2023        0.000239
Name: value, Length: 9, dtype: float64

In [10]:
# Values now represent 2030 instead of 2022
asfr = (
    asfr.reset_index()
    .assign(year_start=2030, year_end=2031)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
India     Female  0.000000   0.019178   2030        2031        0.000000
                  0.019178   0.076712   2030        2031        0.000000
                  0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                                                                  ...   
                  35.000000  40.000000  2030        2031        0.028329
                  30.000000  35.000000  2030        2031        0.068199
                  20.000000  25.000000  2030        2031        0.084126
                  25.000000  30.000000  2030        2031        0.107037
Name: value, Length: 50, dtype: float64

In [11]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import resolve_location
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [12]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = resolve_location(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        # NOTE: Using RELEASE_IDS.GBD_2023 returns an empty dataframe,
        # so we're keeping GBD 2021 forecasts for now
        # TODO: Update to the latest GBD release once it provides 
        # forecasted population data
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [13]:
with gbd_data.quiet_gbd_logs():
    pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2030        2031        1.761904e+05
                  0.019178   0.076712    2030        2031        5.254291e+05
                  0.076712   0.500000    2030        2031                 NaN
                  0.500000   1.000000    2030        2031                 NaN
                                                                     ...     
          Male    80.000000  85.000000   2030        2031        6.771079e+06
                  85.000000  90.000000   2030        2031        2.880948e+06
                  90.000000  95.000000   2030        2031        9.755468e+05
                  95.000000  125.000000  2030        2031        2.694498e+05
Name: value, Length: 50, dtype: float64

In [14]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
India     Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [15]:
pop = pop.fillna(0)

In [16]:
n_births = (pop * asfr).sum()
n_births

19041781.403899632

In [17]:
# NOTE: The stillbirth ratio (SBR) pulled from GBD here will be applied
# to the future population in 2030 (or whatever target year is being
# used). The SBR does not vary much by year, so we just use data for the
# most recent GBD year (defined in YEAR above) and extrapolate that
# value forward to the target year.
with gbd_data.quiet_gbd_logs():
    sbr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio,
        "estimate",
        location.title(),
        years=YEAR,
    ).value
sbr

location  year_start  year_end  parameter  
India     2023        2024      lower_value    0.014030
                                mean_value     0.016334
                                upper_value    0.019572
Name: value, dtype: float64

In [18]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
).squeeze()
sbr

0.0163338791237466

In [19]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

19.352807559651737

## Fertility (technically birth-and-stillbirth) disparities

In [20]:
# TODO: Update this DHS data, and move it into 0100_data_prep? We already
# have some stuff in there to process DHS data.
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        {  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            1: 56_979,
            2: 50_335,
            3: 45_189,
            4: 42_611,
            5: 36_290,
        }
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        {  # Table 8.4 Perinatal mortality
            1: 7_712,
            2: 7_886,
            3: 7_139,
            4: 6_328,
            5: 5_558,
        }
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        {  # Table 8.4 Perinatal mortality
            1: 2_645,
            2: 2_516,
            3: 2_290,
            4: 2_018,
            5: 1_592,
        }
    )

dist_births_and_stillbirths_by_wealth.index.name = "wealth_quintile"

# TODO: Clarify variable names: What does the prefix s_ stand for in
# this notebook? Claude thinks it means "stratified," but that it is
# applied inconsistently and sometimes redundantly with the suffix
# _by_wealth.
s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

wealth_quintile
1    4.688690e+06
2    4.141968e+06
3    3.718514e+06
4    3.506376e+06
5    2.986233e+06
dtype: float64

In [21]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

wealth_quintile
1    4.765275e+06
2    4.209623e+06
3    3.779252e+06
4    3.563648e+06
5    3.035010e+06
dtype: float64

In [22]:
# TODO: Update this with newer GBD data
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths in 2030 for under-1 year olds
# from GBD 2021
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate_per_birth = ntd_deaths / n_births
10_000 * ntd_death_rate_per_birth

2.2442070462613555

In [23]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

8.602793677335196

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [24]:
# TODO: Search for updated folate intake data
if location == "india":
    folate_intake_by_wealth = pd.Series(
        {
            1: 220,
            2: 220,
            3: 220,
            4: 220,
            5: 220,  # NRV is 400 mcg/day
        }
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    folate_intake_by_wealth = pd.Series(
        {
            1: 189,
            2: 198,
            3: 197,
            4: 203,
            5: 208,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    folate_intake_by_wealth = pd.Series(
        {
            1: 166,
            2: 152,
            3: 137,
            4: 350,
            5: 469,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?

folate_intake_by_wealth.index.name = "wealth_quintile"

In [25]:
# TODO: See if we can find better data here
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        {  # it includes wealth stratification, but has a very low threshold for insufficiency
            1: 1,  # so I am assuming that most everyone is in the danger zone for low folate
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        {  # can CHAMPS offer more detail?  Need to infer wealth somehow
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        {  # supplementation studies don't make this easy, but here is a guess
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
    # FIXME: Why is this normalization only applied for Ethiopia? Should
    # it be applied for all locations? Also, Claude thinks the correct
    # normalization would be to divide by the bifth-weighted mean, not
    # the unweighted mean, and that the assert statement below only
    # passes because these values are all 1s.
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth.index.name = "wealth_quintile"
s_dist_deaths_by_wealth

wealth_quintile
1    1
2    1
3    1
4    1
5    1
dtype: int64

In [26]:
s_ntd_death_rate_per_birth = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate_per_birth

wealth_quintile
1    2.244207
2    2.244207
3    2.244207
4    2.244207
5    2.244207
dtype: float64

In [27]:
s_ntd_death_count = s_ntd_death_rate_per_birth * s_births
s_ntd_death_count

wealth_quintile
1    1052.239154
2     929.543478
3     834.511577
4     786.903291
5     670.172500
dtype: float64

In [28]:
assert np.isclose(s_ntd_death_count.sum(), ntd_deaths)

In [29]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

wealth_quintile
1    2981.344270
2    2633.706521
3    2364.449468
4    2229.559324
5    1898.822085
dtype: float64

In [30]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

wealth_quintile
1    4033.583424
2    3563.249998
3    3198.961045
4    3016.462614
5    2568.994585
dtype: float64

In [31]:
# NOTE: NTD risk here means risk of having an "NTD-affected pregnancy",
# which is a stillbirth due to NTD, OR a birth with NTD (not necessarily fatal!)
# See Kirke 1993 ("Maternal plasma folate and vitamin B12 are independent risk factors for neural tube defects")
# where it says: "Early foetal
# deaths (<23 weeks gestation) attributable to NTDs
# were excluded because of the incomplete ascertain-
# ment of such cases and the difficulty of obtaining a
# valid control group."
# This implies that late foetal deaths, roughly equivalent to stillbirths,
# are included.
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc

In [32]:
# TODO: Update these to GBD 2023
# From GBD 2021 using GBD Compare, for year 2021
# NTD incident cases / NTD deaths in <1 year olds
# (all of GBD's incident cases are those who survived birth)
if location == "india":
    ntd_death_to_live_birth_case_ratio = 11_796.34 / 5_492.68
elif location == "nigeria":
    ntd_death_to_live_birth_case_ratio = 21_756.56 / 6_231.21
elif location == "ethiopia":
    ntd_death_to_live_birth_case_ratio = 3_982.63 / 2_255.6

ntd_death_to_live_birth_case_ratio

2.1476474143769524

In [33]:
s_ntd_live_birth_cases = s_ntd_death_count * ntd_death_to_live_birth_case_ratio
s_ntd_live_birth_cases

wealth_quintile
1    2259.838699
2    1996.331647
3    1792.236630
4    1689.990818
5    1439.294238
dtype: float64

In [34]:
s_ntd_affected_pregnancies = s_ntd_stillbirth_count + s_ntd_live_birth_cases

In [35]:
ntd_affected_pregnancy_risk = (
    s_ntd_affected_pregnancies / s_births_and_stillbirths_by_wealth
)
ntd_affected_pregnancy_risk

wealth_quintile
1    0.0011
2    0.0011
3    0.0011
4    0.0011
5    0.0011
dtype: float64

In [36]:
s_ntd_death_or_stillbirth_count / s_births

wealth_quintile
1    0.00086
2    0.00086
3    0.00086
4    0.00086
5    0.00086
dtype: float64

In [37]:
backcalc_rbc(ntd_affected_pregnancy_risk, "daly")

wealth_quintile
1    1037.712291
2    1037.712291
3    1037.712291
4    1037.712291
5    1037.712291
dtype: float64

In [38]:
backcalc_rbc(
    ntd_affected_pregnancy_risk, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9
# NOTE: It's a bit hard to compare, due to units issues. That table reports
# proportions under 151 ng/ml. In our units (nmol/L), that is ~342.
# https://www.wolframalpha.com/input?i=151+ng%2Fml+of+folate+to+nmol%2FL

wealth_quintile
1    808.336224
2    808.336224
3    808.336224
4    808.336224
5    808.336224
dtype: float64

In [39]:
pop

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2030        2031        1.761904e+05
                  0.019178   0.076712    2030        2031        5.254291e+05
                  0.076712   0.500000    2030        2031        0.000000e+00
                  0.500000   1.000000    2030        2031        0.000000e+00
                                                                     ...     
          Male    80.000000  85.000000   2030        2031        6.771079e+06
                  85.000000  90.000000   2030        2031        2.880948e+06
                  90.000000  95.000000   2030        2031        9.755468e+05
                  95.000000  125.000000  2030        2031        2.694498e+05
Name: value, Length: 50, dtype: float64

In [40]:
s_pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
s_pop = s_pop.set_index([c for c in s_pop.columns if c != "value"])
s_pop

value
sex    age_start age_end    pregnant     wealth_quintile              
Female 0.0       0.019178   not_pregnant 1                49649.700497
                                         2                43575.622144
                                         3                38689.356750
                                         4                36440.833935
...                                                                ...
Male   95.0      125.000000 not_pregnant 2                15848.509818
                                         3                16370.176208
                                         4                17265.313670
                                         5                21289.473456

[285 rows x 1 columns]

In [41]:
# WRA only
s_pop = s_pop[
    (s_pop.index.get_level_values("sex") == "Female")
    & (s_pop.index.get_level_values("age_start") >= 15)
    & (s_pop.index.get_level_values("age_end") <= 50)
].copy()

In [42]:
s_daily_vehicle = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/amount/mean/{location}.csv"
)
assert (s_daily_vehicle.vehicle_name == vehicle).all()
s_daily_vehicle = s_daily_vehicle.drop(columns=["vehicle_name"])
s_daily_vehicle = s_daily_vehicle.set_index(
    [c for c in s_daily_vehicle.columns if c != "value"]
).value
s_daily_vehicle

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  185.559631
                            2                  147.531394
                            3                  129.130302
                            4                  127.809856
                                                  ...    
Male    50         125      2                  185.190235
                            3                  176.087660
                            4                  170.708637
                            5                  133.137342
Name: value, Length: 50, dtype: float64

In [43]:
from lsff_utils import data_processing

In [44]:
s_daily_vehicle = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, s_daily_vehicle)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
s_daily_vehicle

wealth_quintile
1    211.695632
2    174.262141
3    162.011968
4    162.387602
5    127.329933
Name: value, dtype: float64

In [45]:
any_consumers = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
assert (any_consumers.vehicle_name == vehicle).all()
any_consumers = any_consumers.drop(columns=["vehicle_name"])
any_consumers = any_consumers.set_index(
    [c for c in any_consumers.columns if c != "value"]
).value
any_consumers

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  0.919076
                            2                  0.887756
                            3                  0.875160
                            4                  0.874946
                                                 ...   
Male    50         125      2                  0.975934
                            3                  0.976151
                            4                  0.985469
                            5                  0.987381
Name: value, Length: 50, dtype: float64

In [46]:
any_consumers = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, any_consumers)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
any_consumers

wealth_quintile
1    0.990320
2    0.971073
3    0.963712
4    0.985178
5    0.988109
Name: value, dtype: float64

In [47]:
s_daily_vehicle_among_consumers = s_daily_vehicle / any_consumers
s_daily_vehicle_among_consumers

wealth_quintile
1    213.764832
2    179.453099
3    168.112361
4    164.830721
5    128.862297
Name: value, dtype: float64

In [48]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.125

In [49]:
intervention_concentration_mcg_per_gram = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.drop(columns=["vehicle_name"])
    .set_index("scenario")
    .value
)
intervention_concentration_mcg_per_gram

scenario
intervention    1.3
Name: value, dtype: float64

In [50]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()

In [51]:
if location == "india" and vehicle == "rice":
    # Confusingly, our baseline scenario (our best guess about the present)
    # is *not* a good guess about 2019-2020 (which is when our baseline folate estimate is from),
    # because this program has rolled out entirely since then:
    # In the phase-I of the roll out, the fortified rice was introduced in the social welfare schemes such as Integrated Child Development Scheme (ICDS)
    # and Pradhan Mantri Poshan Shakti Nirman (PM POSHAN, earlier known as the National Program of Mid-Day Meal in Schools)
    # throughout India during 2021–22 [18].
    # Phase-II has covered aspirational and high burden districts for anemia (total 291 districts) under Public Distribution System (PDS) and other welfare schemes,
    # in addition to Phase-I districts, by March 2023 [18].
    # All the remaining districts in India will be covered in Phase III by March 2024 [19].
    # ~ https://pmc.ncbi.nlm.nih.gov/articles/PMC11305529/
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline.assign(value=0)
else:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline

In [52]:
df_eff_fort_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()
df_eff_fort_intervention = df_eff_fort_intervention.drop(columns=["vehicle_name"])
df_eff_fort_intervention

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0,5,1,0.295892,intervention
1,Female,0,5,2,0.299771,intervention
2,Female,0,5,3,0.280152,intervention
3,Female,0,5,4,0.241038,intervention
...,...,...,...,...,...,...
46,Male,50,125,2,0.367314,intervention
47,Male,50,125,3,0.330535,intervention
48,Male,50,125,4,0.281447,intervention
49,Male,50,125,5,0.146075,intervention


In [53]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,1,1.144883e+07
41,Female,15.0,20.0,2,1.300234e+07
42,Female,15.0,20.0,3,1.357840e+07
43,Female,15.0,20.0,4,1.349499e+07
...,...,...,...,...,...
71,Female,45.0,50.0,2,7.200663e+06
72,Female,45.0,50.0,3,7.650878e+06
73,Female,45.0,50.0,4,8.210394e+06
74,Female,45.0,50.0,5,8.760835e+06


In [54]:
if "sex" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.sex == "Female")
    ]

if "age_start" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.age_start >= 15)
        & (df_eff_fort_baseline_2019_2020.age_end <= 50)
    ]

In [55]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [56]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population) * (
        1
        if "scenario" not in effective_fort.columns
        else effective_fort.scenario.nunique()
    )
    group_cols = [c for c in ["scenario", "wealth_quintile"] if c in merged]
    return merged.groupby(group_cols).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [57]:
df_eff_fort_baseline_2019_2020 = aggregate_using_population(
    df_eff_fort_baseline_2019_2020
)
df_eff_fort_baseline_2019_2020

    wealth_quintile vehicle_name     sex  age_start_fort  age_end_fort  \
0                 1         rice  Female              15            30   
1                 1         rice  Female              15            30   
2                 1         rice  Female              15            30   
10                1         rice  Female              30            50   
..              ...          ...     ...             ...           ...   
66                5         rice  Female              30            50   
67                5         rice  Female              30            50   
68                5         rice  Female              30            50   
69                5         rice  Female              30            50   

    value_fort  index  age_start_pop  age_end_pop     value_pop  
0            0     40           15.0         20.0  1.144883e+07  
1            0     45           20.0         25.0  1.161927e+07  
2            0     50           25.0         30.0  1.097286e+

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [58]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

     wealth_quintile vehicle_name     sex  age_start_fort  age_end_fort  \
14                 1         rice  Female              15            30   
15                 1         rice  Female              15            30   
16                 1         rice  Female              15            30   
24                 1         rice  Female              30            50   
..               ...          ...     ...             ...           ...   
164                5         rice  Female              30            50   
165                5         rice  Female              30            50   
166                5         rice  Female              30            50   
167                5         rice  Female              30            50   

     value_fort  index  age_start_pop  age_end_pop     value_pop  
14     0.340944     40           15.0         20.0  1.144883e+07  
15     0.340944     45           20.0         25.0  1.161927e+07  
16     0.340944     50           25.0         30

wealth_quintile
1    0.354760
2    0.357360
3    0.319993
4    0.282220
5    0.158611
dtype: float64

In [59]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

       sex  age_start_fort  age_end_fort  wealth_quintile  value_fort  \
0   Female              15            30                1    0.340944   
1   Female              15            30                1    0.340944   
2   Female              15            30                1    0.340944   
10  Female              30            50                1    0.368677   
..     ...             ...           ...              ...         ...   
66  Female              30            50                5    0.151782   
67  Female              30            50                5    0.151782   
68  Female              30            50                5    0.151782   
69  Female              30            50                5    0.151782   

        scenario  index  age_start_pop  age_end_pop     value_pop  
0   intervention     40           15.0         20.0  1.144883e+07  
1   intervention     45           20.0         25.0  1.161927e+07  
2   intervention     50           25.0         30.0  1.097286e+07

scenario      wealth_quintile
intervention  1                  0.354760
              2                  0.357360
              3                  0.319993
              4                  0.282220
              5                  0.158611
dtype: float64

In [60]:
RBC_baseline = backcalc_rbc(ntd_affected_pregnancy_risk, "crider")
RBC_baseline

wealth_quintile
1    808.336224
2    808.336224
3    808.336224
4    808.336224
5    808.336224
dtype: float64

In [61]:
# TODO: Document this value better
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [62]:
# TODO: Determine whether we still want the same fortification
# scenarios, and update baseline values
# Delete fortification effect baked into our baseline folate estimate.
# In non-India locations, this is going to be zero.
# For India, our current source for baseline folate is very rough,
# but it does appear to be from before the fortification program (2019-2020).
s_zero_folate = folate_intake_by_wealth - (
    df_eff_fort_baseline_2019_2020
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)

In [63]:
s_baseline_folate = s_zero_folate + (
    df_eff_fort_baseline
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_baseline_folate

wealth_quintile
1    236.114994
2    233.627474
3    231.431397
4    229.885180
5    224.343283
dtype: float64

In [64]:
s_intervention_folate = s_zero_folate + (
    df_eff_fort_intervention
    * s_daily_vehicle_among_consumers
    * intervention_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_intervention_folate

scenario      wealth_quintile
intervention  1                  387.595937
              2                  361.725733
              3                  338.886525
              4                  322.805868
              5                  265.170142
dtype: float64

In [65]:
zero_folate_pct_decrease = (
    folate_intake_by_wealth - s_zero_folate
) / folate_intake_by_wealth

In [66]:
baseline_folate_pct_increase_from_zero = (
    s_baseline_folate - s_zero_folate
) / folate_intake_by_wealth
baseline_folate_pct_increase_from_zero

wealth_quintile
1    0.073250
2    0.061943
3    0.051961
4    0.044933
5    0.019742
dtype: float64

In [67]:
intevention_folate_pct_increase_from_zero = (
    s_intervention_folate - s_zero_folate
) / folate_intake_by_wealth
intevention_folate_pct_increase_from_zero

scenario      wealth_quintile
intervention  1                  0.761800
              2                  0.644208
              3                  0.540393
              4                  0.467299
              5                  0.205319
dtype: float64

In [68]:
# TODO: What is this hardcoded (1 + ((6 / 10) * pct_decrease) in this
# cell and the following? Claude says this line (and the subsequent
# cells) is converting a change in dietary folate intake into a change
# in RBC folate concentration using a proportional (elasticity) model
# with an elasticity of 0.6 (e.g. a 10% rise in dietary folate intake
# would lead to a 6% rise in RBC folate concentration). But the value of
# 0.6 is not documented anywhere. Does it come from Crider et al.?
RBC_zero = RBC_baseline / (1 + ((6 / 10) * zero_folate_pct_decrease))
RBC_zero

wealth_quintile
1    808.336224
2    808.336224
3    808.336224
4    808.336224
5    808.336224
dtype: float64

In [69]:
RBC_baseline = RBC_zero * (1 + ((6 / 10) * baseline_folate_pct_increase_from_zero))
RBC_baseline

wealth_quintile
1    843.862588
2    838.378718
3    833.537348
4    830.128630
5    817.911223
dtype: float64

In [70]:
RBC_intervention = RBC_zero * (
    1 + ((6 / 10) * intevention_folate_pct_increase_from_zero)
)
RBC_intervention

scenario      wealth_quintile
intervention  1                  1177.810407
              2                  1120.778161
              3                  1070.427910
              4                  1034.977244
              5                   907.916212
dtype: float64

In [71]:
# TODO: Document this function better. I think it's the inverse of the
# backcalc_rbc function above
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds)  # TODO: better transformation
    return p

In [72]:
# NOTE: All rates here are per birth!
s_ntd_affected_pregnancy_rate_zero = calc_ntd_pr(RBC_zero, "crider")
10_000 * s_ntd_affected_pregnancy_rate_zero

wealth_quintile
1    11.010811
2    11.010811
3    11.010811
4    11.010811
5    11.010811
dtype: float64

In [73]:
s_ntd_affected_pregnancy_rate_baseline = calc_ntd_pr(RBC_baseline, "crider")
10_000 * s_ntd_affected_pregnancy_rate_baseline

wealth_quintile
1    10.234434
2    10.348499
3    10.450888
4    10.523946
5    10.792581
dtype: float64

In [74]:
s_ntd_affected_pregnancy_rate_intervention = calc_ntd_pr(RBC_intervention, "crider")
10_000 * s_ntd_affected_pregnancy_rate_intervention

scenario      wealth_quintile
intervention  1                  5.806275
              2                  6.317458
              3                  6.830904
              4                  7.233416
              5                  9.037493
dtype: float64

In [75]:
s_ntd_affected_pregnancies_zero = (
    s_ntd_affected_pregnancy_rate_zero * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_zero

wealth_quintile
1    5246.953936
2    4635.136215
3    4161.262947
4    3923.865884
5    3341.791859
dtype: float64

In [76]:
s_ntd_affected_pregnancies_baseline = (
    s_ntd_affected_pregnancy_rate_baseline * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_baseline

wealth_quintile
1    4876.988993
2    4356.327831
3    3949.653721
4    3750.364321
5    3275.558668
dtype: float64

In [77]:
s_ntd_affected_pregnancies_intervention = (
    s_ntd_affected_pregnancy_rate_intervention * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_intervention

scenario      wealth_quintile
intervention  1                  2766.849667
              2                  2659.411546
              3                  2581.570609
              4                  2577.734970
              5                  2742.887884
dtype: float64

In [78]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_affected_pregnancies_zero.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="zero")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_affected_pregnancies_baseline.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        *[
            s_ntd_affected_pregnancies_intervention.loc[intervention_scenario]
            .rename("value")
            .to_frame()
            .assign(entity="ntd", scenario=intervention_scenario)
            .set_index(["entity", "scenario"], append=True)
            .value
            for intervention_scenario in intervention_scenarios
        ],
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario    
1                ntd     zero            5246.953936
2                ntd     zero            4635.136215
3                ntd     zero            4161.262947
4                ntd     zero            3923.865884
                                            ...     
2                ntd     intervention    2659.411546
3                ntd     intervention    2581.570609
4                ntd     intervention    2577.734970
5                ntd     intervention    2742.887884
Name: value, Length: 15, dtype: float64

In [79]:
(
    ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "baseline"
    ].droplevel("scenario")
    - ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "intervention"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       2110.139327
2                ntd       1696.916285
3                ntd       1368.083111
4                ntd       1172.629351
5                ntd        532.670784
Name: value, dtype: float64

In [80]:
(
    ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "zero"
    ].droplevel("scenario")
    - ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "baseline"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       369.964943
2                ntd       278.808384
3                ntd       211.609226
4                ntd       173.501563
5                ntd        66.233191
Name: value, dtype: float64

In [81]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [82]:
ntd_deaths_and_stillbirths_by_scenario = ntd_cases_by_scenario * (
    s_ntd_death_or_stillbirth_count / s_ntd_affected_pregnancies
)

In [83]:
(
    ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "baseline"
    ].droplevel("scenario")
    - ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "intervention"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       1623.950750
2                ntd       1305.936740
3                ntd       1052.868674
4                ntd        902.448615
5                ntd        409.940286
dtype: float64

In [84]:
(
    ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "zero"
    ].droplevel("scenario")
    - ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "baseline"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       284.722833
2                ntd       214.569284
3                ntd       162.853209
4                ntd       133.525777
5                ntd        50.972672
dtype: float64

In [85]:
# For calculating YLLs
with gbd_data.quiet_gbd_logs():
    tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [86]:
# NOTE: Treating stillbirths as a death!
yll_per_stillbirth_or_death = float(tmrle.iloc[0])
yll_per_stillbirth_or_death

89.95803974533831

In [87]:
ylls_by_scenario = (
    ntd_deaths_and_stillbirths_by_scenario * yll_per_stillbirth_or_death
).rename("value")
ylls_by_scenario

wealth_quintile  entity  scenario    
1                ntd     zero            363252.788867
2                ntd     zero            320895.928809
3                ntd     zero            288089.125399
4                ntd     zero            271653.847670
                                             ...      
2                ntd     intervention    184114.187482
3                ntd     intervention    178725.167952
4                ntd     intervention    178459.622114
5                ntd     intervention    189893.352485
Name: value, Length: 15, dtype: float64

In [88]:
path = f"./results/{location}/{vehicle}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)